# 🔬 Multilingual & Code-Switched Fake News Detection
### End-to-End Google Colab / Kaggle T4 GPU Training & Benchmark Pipeline

**Models Evaluated:**
1. `ai4bharat/indic-bert` (IndicBERT - 12M params)
2. `google/muril-base-cased` (MuRIL - 110M params)
3. `xlm-roberta-base` (Multilingual RoBERTa - 278M params)
4. `bert-base-uncased` (English Monolingual Baseline - 110M params)

**Objective:**
Scientifically evaluate cross-lingual transfer and subword tokenization efficacy across **Hindi (Devanagari)**, **Hinglish (Code-Switched Roman)**, and **English**.

## Step 1: GPU Diagnostics & Dependency Installation

In [ ]:
# Check CUDA GPU availability
!nvidia-smi

# Install core PyTorch, Transformers, Datasets, and Evaluation libraries
!pip install --quiet --upgrade torch torchvision torchaudio
!pip install --quiet transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn onnx onnxruntime

## Step 2: Mount Google Drive & Set Up Directories

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

WORKSPACE_DIR = "/content/multilingual_fake_news"
os.makedirs(f"{WORKSPACE_DIR}/data", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/results", exist_ok=True)
%cd {WORKSPACE_DIR}

## Step 3: Dataset Ingestion & Preprocessing
Upload your dataset CSV or use the built-in seed dataset.

In [ ]:
import re
import unicodedata

# Preprocessing functions
URL_PATTERN = re.compile(r"(?:https?://|www\.)[a-zA-Z0-9.\-_~:/?#\[\]@!$&'()*+,;=%]+")
PHONE_PATTERN = re.compile(r"(?:\+91[\-\s]?)?[6-9]\d{9}|\b\d{3}[-.\s]??\d{3}[-.\s]??\d{4}\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
ZERO_WIDTH_CHARS = re.compile(r"[\u200B\u200C\u200D\u200E\u200F\uFEFF]")
ELONGATED_PATTERN = re.compile(r"(.)\1{2,}")

def preprocess_text(text):
    if not isinstance(text, str): return "", "unknown"
    text = unicodedata.normalize("NFC", text)
    text = ZERO_WIDTH_CHARS.sub("", text)
    text = URL_PATTERN.sub("[URL]", text)
    text = PHONE_PATTERN.sub("[PHONE]", text)
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    text = ELONGATED_PATTERN.sub(r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    # Detect script
    dev_count = len(re.findall(r"[\u0900-\u097F]", text))
    lat_count = len(re.findall(r"[a-zA-Z]", text))
    total = dev_count + lat_count
    if total == 0: script = "unknown"
    elif dev_count / total >= 0.6: script = "hindi_devanagari"
    elif lat_count / total >= 0.6: 
        hinglish_words = {"hai", "hain", "ko", "se", "me", "par", "ke", "ki", "ka", "bhai", "karo", "nahi"}
        overlap = len(set(text.lower().split()).intersection(hinglish_words))
        script = "hinglish_roman" if overlap >= 2 else "english_latin"
    else: script = "code_mixed"
    return text, script

# Check if user provided dataset exists
data_path = "/content/multilingual_fake_news/data/dataset.csv"
if not os.path.exists(data_path):
    print("Creating sample balanced dataset...")
    sample_records = [
        ("विश्व स्वास्थ्य संगठन ने कोरोना वायरस के लिए नई गाइडलाइन जारी की।", 0, "hindi"),
        ("सावधान! कल रात 12 बजे से पृथ्वी पर कॉस्मिक किरणें आएंगी मोबाइल बंद रखें।", 1, "hindi"),
        ("RBI ne repo rate 6.5% par unchanged rakhne ka decision liya hai.", 0, "hinglish"),
        ("Govt PM Yojna ke under sabhi ko 5000 rs de rahi hai, link pe click karo!", 1, "hinglish"),
        ("NASA confirmed detection of atmospheric water vapor on exoplanet K2-18b.", 0, "english"),
        ("Boiled garlic water cures pneumonia and prevents viral infections in 12 hours.", 1, "english")
    ] * 20  # Replicate for demonstration
    df = pd.DataFrame(sample_records, columns=["text", "label", "language"])
    df.to_csv(data_path, index=False)

df = pd.read_csv(data_path)
cleaned_texts, scripts = zip(*[preprocess_text(t) for t in df["text"]])
df["cleaned_text"] = cleaned_texts
df["detected_script"] = scripts
print(f"Loaded {len(df)} samples. Class balance:\n{df['label'].value_counts()}")

## Step 4: Stratified Train / Val / Test Partitioning

In [ ]:
from sklearn.model_selection import train_test_split

# 80% Train, 10% Validation, 10% Test
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(f"Train samples: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## Step 5: 4-Model Fine-Tuning Execution Loop

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

MODELS_CONFIG = {
    "indic-bert": {"name": "l3cube-pune/hindi-bert", "alt_name": "ai4bharat/indic-bert", "lr": 3e-5, "batch": 16},
    "muril": {"name": "google/muril-base-cased", "lr": 2e-5, "batch": 16},
    "xlm-roberta": {"name": "xlm-roberta-base", "lr": 1.5e-5, "batch": 16},
    "bert-base-uncased": {"name": "bert-base-uncased", "lr": 2e-5, "batch": 16}
}

class TorchDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "macro_f1": f1, "precision": prec, "recall": rec}

benchmark_results = {}

for key, cfg in MODELS_CONFIG.items():
    model_identifier = cfg["name"]
    print(f"\n{'='*20} Fine-Tuning: {key} ({model_identifier}) {'='*20}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_identifier)
        model = AutoModelForSequenceClassification.from_pretrained(model_identifier, num_labels=2).to(device)
    except Exception as load_err:
        print(f"[Notice] Could not load gated/restricted model '{model_identifier}': {load_err}")
        alt = cfg.get("alt_name", "bert-base-multilingual-cased")
        print(f"Switching to alternative model: {alt}")
        try:
            tokenizer = AutoTokenizer.from_pretrained(alt)
            model = AutoModelForSequenceClassification.from_pretrained(alt, num_labels=2).to(device)
        except Exception as alt_err:
            print(f"Skipping {key} due to access restriction: {alt_err}")
            continue
    
    train_ds = TorchDataset(train_df["cleaned_text"].tolist(), train_df["label"].tolist(), tokenizer)
    val_ds = TorchDataset(val_df["cleaned_text"].tolist(), val_df["label"].tolist(), tokenizer)
    test_ds = TorchDataset(test_df["cleaned_text"].tolist(), test_df["label"].tolist(), tokenizer)
    
    training_args = TrainingArguments(
        output_dir=f"./checkpoints/{key}",
        num_train_epochs=3,
        per_device_train_batch_size=cfg["batch"],
        learning_rate=cfg["lr"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    # Hold-out test set evaluation
    preds_raw = trainer.predict(test_ds)
    y_preds = np.argmax(preds_raw.predictions, axis=1)
    y_trues = test_df["label"].values
    
    acc = accuracy_score(y_trues, y_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_trues, y_preds, average="macro", zero_division=0)
    cm = confusion_matrix(y_trues, y_preds).tolist()
    
    benchmark_results[key] = {
        "accuracy": round(float(acc), 4),
        "macro_f1": round(float(f1), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "confusion_matrix": cm
    }
    
    # Save model and tokenizer
    save_path = f"./checkpoints/{key}/best_model"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Successfully saved {key} to {save_path}")

## Step 6: Visualizing Benchmark Comparison & Research Insights

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame(benchmark_results).T
print("=== Model Benchmark Comparison Table ===")
display(results_df[["accuracy", "macro_f1", "precision", "recall"]])

# Export JSON and CSV
with open("./results/benchmark_results.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)
results_df.to_csv("./results/benchmark_results.csv")

# Plotting Comparison Bar Chart
plt.figure(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35
plt.bar(x - width/2, results_df["accuracy"], width, label="Accuracy", color="#4361ee")
plt.bar(x + width/2, results_df["macro_f1"], width, label="Macro-F1", color="#4cc9f0")
plt.xticks(x, results_df.index, fontsize=11, fontweight="bold")
plt.ylabel("Score (0.0 to 1.0)")
plt.title("Fake News Sequence Classification: Multilingual vs English Baseline")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Step 7: Export Checkpoints to Downloadable Archive
Package fine-tuned weights and benchmark results for the FastAPI Backend.

In [ ]:
!zip -r multilingual_checkpoints.zip checkpoints/ results/
from google.colab import files
files.download("multilingual_checkpoints.zip")
print("Download started! Unpack this archive into your backend repository.")